# Chapter 9 — The Mean Value Inequality, Contractions and Newton's Method

Companion notebook to Chapter 9 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, hessian

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
op = lambda A: torch.linalg.matrix_norm(A, ord=2).item()      # spectral norm
print(torch.__version__)

2.15.0+cu132


## 1. The mean value inequality

Example 9.1 and Exercise 9.1: no mean value equality for curves, but the
displacement is bounded by speed times duration. Theorem 9.3 and Corollary
9.4 on a random map: the Lipschitz bound, and the two estimates of the error
of the tangent affine map.

In [2]:
circle = lambda t: torch.stack([torch.cos(t), torch.sin(t)])
helix = lambda t: torch.stack([torch.cos(t), torch.sin(t), t])
two_pi = torch.tensor(2 * math.pi); zero = torch.tensor(0.0)
print("circle: displacement", (circle(two_pi) - circle(zero)).norm().item(),
      "  speed * duration", 2 * math.pi)
d = helix(two_pi) - helix(zero)
speeds = torch.stack([jacrev(helix)(t).norm() for t in torch.linspace(0, 2 * math.pi, 50)])
print("helix : displacement", d.tolist(), " norm", d.norm().item(),
      "  speed", speeds[0].item(), "  bound", (speeds.max() * two_pi).item())
assert torch.allclose(speeds, torch.full_like(speeds, math.sqrt(2)))
assert d.norm() <= math.sqrt(2) * two_pi
# no tau with c(2 pi) - c(0) = 2 pi c'(tau): the first two components never vanish together
ts = torch.linspace(0, 2 * math.pi, 100001)
assert (torch.sin(ts) ** 2 + torch.cos(ts) ** 2).min() > 0.99

torch.manual_seed(1)
A1, A2 = torch.randn(5, 4), torch.randn(3, 5)
f = lambda x: A2 @ torch.tanh(A1 @ x)
a, b = torch.randn(4), torch.randn(4)
seg = [a + t * (b - a) for t in torch.linspace(0, 1, 2001)]
K = max(op(jacrev(f)(x)) for x in seg)
print("\n|f(b) - f(a)| =", (f(b) - f(a)).norm().item(), " <= K |b - a| =", K * (b - a).norm().item())
assert (f(b) - f(a)).norm() <= K * (b - a).norm()

Ja = jacrev(f)(a)
err = (f(b) - f(a) - Ja @ (b - a)).norm().item()
sup = max(op(jacrev(f)(x) - Ja) for x in seg)
print("error of the tangent affine map:", err, " <= sup|Df - Df(a)| |b - a| =", sup * (b - a).norm().item())
assert err <= sup * (b - a).norm().item()

circle: displacement 2.4492935982947064e-16   speed * duration 6.283185307179586


helix : displacement [0.0, -2.4492935982947064e-16, 6.283185307179586]  norm 6.283185307179586   speed 1.4142135623730951   bound 8.885765876316732



|f(b) - f(a)| = 1.8672573586526544  <= K |b - a| = 13.523756014940671


error of the tangent affine map: 1.545805570922831  <= sup|Df - Df(a)| |b - a| = 14.51637380775129


In [3]:
# Corollary 9.4(4): if Df is L-Lipschitz, the error is at most (L/2)|b - a|^2.
# For f(x) = A2 tanh(A1 x): |Df(x) - Df(y)| <= |A2| max|tanh''| |A1|^2 |x - y|.
tanh2_max = 4 / (3 * math.sqrt(3))               # max of |tanh''| = 0.7698...
Lc = op(A2) * tanh2_max * op(A1) ** 2
worst = 0.0
for _ in range(300):
    x, y = torch.randn(4), torch.randn(4)
    lhs = op(jacrev(f)(x) - jacrev(f)(y)); assert lhs <= Lc * (x - y).norm().item() + 1e-12
    e = (f(y) - f(x) - jacrev(f)(x) @ (y - x)).norm().item()
    assert e <= 0.5 * Lc * (y - x).norm().item() ** 2 + 1e-12
    worst = max(worst, e / (0.5 * Lc * (y - x).norm().item() ** 2))
print("L =", round(Lc, 3), "  largest ratio error / ((L/2)|b - a|^2) over 300 pairs:", round(worst, 4))

L = 41.403   largest ratio error / ((L/2)|b - a|^2) over 300 pairs: 0.171


### Smoothness and Lipschitz gradients; Lipschitz constant of a layer

Proposition 9.5: for a `C^2` function, the largest `|eigenvalue|` of the
Hessian operator over the domain is the Lipschitz constant of the gradient.
Exercise 9.2: the layer `tanh(W x + b)` with `W = [[3, 0], [4, 5]]` is
Lipschitz with constant `|W| = 3 sqrt(5)`, and the constant is attained.

In [4]:
fs = lambda x: torch.log(torch.cosh(x)).sum() * 3 + 0.5 * (x ** 2).sum() + torch.sin(x[0] * x[1])
M = torch.tensor([[2.0, 1.0], [1.0, 2.0]])                       # an inner product
grad = lambda x: torch.linalg.solve(M, jacrev(fs)(x))
mnorm = lambda v: math.sqrt(v @ M @ v)
torch.manual_seed(2)
hess_norm = lambda x: torch.linalg.eigvals(torch.linalg.solve(M, hessian(fs)(x))).real.abs().max().item()
tight = 0.0
for _ in range(100):
    x, y = 1.5 * torch.randn(2), 1.5 * torch.randn(2)
    L_seg = max(hess_norm(x + t * (y - x)) for t in torch.linspace(0, 1, 101))
    ratio = mnorm(grad(x) - grad(y)) / mnorm(x - y)
    assert ratio <= L_seg * (1 + 1e-9)        # mean value inequality for the gradient field
    tight = max(tight, ratio / L_seg)
# and the bound is reached in the limit of nearby points along the top eigenvector
x = torch.tensor([0.2, -0.4])
lam, vec = torch.linalg.eig(torch.linalg.solve(M, hessian(fs)(x)))
v = vec[:, lam.real.abs().argmax()].real
local = mnorm(grad(x + 1e-6 * v) - grad(x)) / mnorm(1e-6 * v)
print("largest ratio / (max |eigenvalue| on the segment) over 100 pairs:", round(tight, 4))
print("|eigenvalue|_max at a point:", round(hess_norm(x), 5), "  local ratio along its eigenvector:", round(local, 5))
assert abs(local - hess_norm(x)) < 1e-4

W = torch.tensor([[3.0, 0.0], [4.0, 5.0]]); bvec = torch.tensor([0.3, -0.2])
layer = lambda x: torch.tanh(W @ x + bvec)
x0 = torch.linalg.solve(W, -bvec)                                  # where W x + b = 0
print("|W| =", op(W), "  |D layer| at the point where Wx + b = 0:", op(jacrev(layer)(x0)))
assert abs(op(W) - 3 * math.sqrt(5)) < 1e-12 and abs(op(jacrev(layer)(x0)) - op(W)) < 1e-12
for _ in range(200):
    x, y = torch.randn(2), torch.randn(2)
    assert (layer(x) - layer(y)).norm() <= op(W) * (x - y).norm() + 1e-12
    assert (layer(layer(x)) - layer(layer(y))).norm() <= 45 * (x - y).norm() + 1e-12

largest ratio / (max |eigenvalue| on the segment) over 100 pairs: 0.9408
|eigenvalue|_max at a point: 2.75013   local ratio along its eigenvector: 2.75013
|W| = 6.70820393249937   |D layer| at the point where Wx + b = 0: 6.70820393249937


## 2. Iterations and contractions

Exercise 9.5 (`x = cos x`), Theorem 9.7 (local rate given by `|Dg(a*)|`),
Theorem 9.9 (gradient descent is a contraction) on Exercise 9.6, and the
remark on which minima a learning rate allows.

In [5]:
kappa = math.sin(1.0)
a, traj = 1.0, [1.0]
for _ in range(200):
    a = math.cos(a); traj.append(a)
astar = traj[-1]
print("fixed point:", astar, "  kappa = sin 1 =", round(kappa, 4))
assert abs(astar - 0.739085) < 1e-6
need = math.ceil(math.log(1e-6 * (1 - kappa) / abs(traj[1] - traj[0])) / math.log(kappa))
actual = next(k for k, v in enumerate(traj) if abs(v - astar) < 1e-6)
print("iterations guaranteed by the a priori bound:", need, "   actually needed:", actual)
assert need == 87 and actual < need
for k in (1, 5, 20):                                   # the a priori bound holds
    assert abs(traj[k] - astar) <= kappa ** k / (1 - kappa) * abs(traj[1] - traj[0])
rates = [abs(traj[k + 1] - astar) / abs(traj[k] - astar) for k in range(20, 30)]
print("observed rate near the fixed point:", round(rates[-1], 4), "  sin(a*) =", round(math.sin(astar), 4))
assert abs(rates[-1] - math.sin(astar)) < 1e-3 and abs(math.sin(astar) - 0.6736) < 1e-4

fixed point: 0.7390851332151607   kappa = sin 1 = 0.8415
iterations guaranteed by the a priori bound: 87    actually needed: 32
observed rate near the fixed point: 0.6736   sin(a*) = 0.6736


In [6]:
# Exercise 9.6 and Theorem 9.9: f = (x1^2 + x2^2)/2 + 3 log cosh x2, mu = 1, L = 4
f6 = lambda x: 0.5 * (x ** 2).sum() + 3 * torch.log(torch.cosh(x[1]))
ev = torch.stack([torch.linalg.eigvalsh(hessian(f6)(3 * torch.randn(2))) for _ in range(500)])
print("Hessian eigenvalues on a sample: min", ev.min().item(), " max", round(ev.max().item(), 4),
      "  at x2 = 0:", torch.linalg.eigvalsh(hessian(f6)(torch.zeros(2))).tolist())
assert ev.min() >= 1 - 1e-12 and ev.max() <= 4 + 1e-12
mu, Lc = 1.0, 4.0
kap = lambda eta: max(abs(1 - eta * mu), abs(1 - eta * Lc))
assert abs(kap(0.25) - 0.75) < 1e-12 and abs(kap(0.4) - 0.6) < 1e-12

for eta in (0.25, 0.4, 0.45):
    g = lambda x: x - eta * jacrev(f6)(x)
    worst = 0.0
    for _ in range(300):                               # g is kappa(eta)-Lipschitz
        x, y = 3 * torch.randn(2), 3 * torch.randn(2)
        worst = max(worst, ((g(x) - g(y)).norm() / (x - y).norm()).item())
    x = torch.tensor([2.0, -3.0]); errs = [x.norm().item()]
    for _ in range(40):
        x = g(x); errs.append(x.norm().item())         # the minimizer is 0
    assert worst <= kap(eta) + 1e-12
    assert all(errs[k] <= kap(eta) ** k * errs[0] + 1e-15 for k in range(41))
    print(f"eta = {eta}: kappa = {kap(eta):.2f}  largest observed |g(x)-g(y)|/|x-y| = {worst:.4f}"
          f"  error after 40 steps = {errs[-1]:.2e}")

Hessian eigenvalues on a sample: min 1.0  max 3.9998   at x2 = 0: [1.0, 4.0]


eta = 0.25: kappa = 0.75  largest observed |g(x)-g(y)|/|x-y| = 0.7500  error after 40 steps = 2.01e-05


eta = 0.4: kappa = 0.60  largest observed |g(x)-g(y)|/|x-y| = 0.6000  error after 40 steps = 2.85e-09


eta = 0.45: kappa = 0.80  largest observed |g(x)-g(y)|/|x-y| = 0.5811  error after 40 steps = 4.47e-05


In [7]:
# what the learning rate allows: a non-convex loss with two minima of different sharpness
loss = lambda x: (x ** 2 - 1) ** 2 * (1 + 2.0 * torch.sigmoid(6 * x))
d1 = lambda x: jacrev(loss)(torch.tensor(x)).item()
d2 = lambda x: hessian(loss)(torch.tensor(x)).item()
print("curvature at the minimum x = -1:", round(d2(-1.0), 3), "   at x = +1:", round(d2(1.0), 3))
sharp, flat = d2(1.0), d2(-1.0)
assert abs(d1(1.0)) < 1e-12 and abs(d1(-1.0)) < 1e-12 and sharp > flat > 0

def run(x, eta, steps=400):
    for _ in range(steps):
        x = x - eta * d1(x)
    return x
eta = 0.5 * (2 / sharp + 2 / flat)                     # too large for the sharp minimum only
print(f"2/sharp = {2 / sharp:.4f} < eta = {eta:.4f} < 2/flat = {2 / flat:.4f}")
print("started next to the sharp minimum:", round(run(1.0 + 1e-6, eta), 6),
      "   next to the flat one:", round(run(-1.0 + 1e-3, eta), 6))
assert abs(run(-1.0 + 1e-3, eta) + 1) < 1e-9           # the flat minimum attracts
assert abs(run(1.0 + 1e-6, eta) - 1) > 1e-3            # the sharp one repels
assert abs(run(1.0 + 1e-3, 1.0 / sharp) - 1) < 1e-9    # with a smaller step it attracts

curvature at the minimum x = -1: 8.04    at x = +1: 23.96
2/sharp = 0.0835 < eta = 0.1661 < 2/flat = 0.2488


started next to the sharp minimum: -1.0    next to the flat one: -1.0


### A layer defined by a fixed point

`z* = tanh(W z* + U x)` with `|W| < 1`: the iteration converges at the rate
`|W|^k` from any start, to a fixed point that does not depend on the start.

In [8]:
torch.manual_seed(3)
Wd = torch.randn(6, 6); Wd = 0.8 * Wd / op(Wd); Ud = torch.randn(6, 3); xin = torch.randn(3)
gz = lambda z: torch.tanh(Wd @ z + Ud @ xin)
z1, z2 = torch.zeros(6), 5 * torch.randn(6)
gaps = []
for _ in range(120):
    z1, z2 = gz(z1), gz(z2); gaps.append((z1 - z2).norm().item())
print("distance between two runs after 1, 10, 50 steps:", [f"{gaps[k]:.2e}" for k in (0, 9, 49)])
assert all(gaps[k + 1] <= 0.8 * gaps[k] + 1e-15 for k in range(60))
assert (gz(z1) - z1).norm() < 1e-12

distance between two runs after 1, 10, 50 steps: ['2.57e+00', '3.60e-03', '5.73e-15']


## 3. Newton's method for equations

Example 9.13 (square roots, errors squared at each step), Exercise 9.8 (a
system), Proposition 9.11 (invariance), Theorem 9.12 (the constant
`beta L`), Theorem 9.15 (Kantorovich) and the pitfall on `arctan`.

In [9]:
x, errs = 1.0, []
for _ in range(5):
    x = 0.5 * (x + 2 / x); errs.append(abs(x - math.sqrt(2)))
print("errors:", [f"{e:.2e}" for e in errs])
assert abs(errs[0] - 8.6e-2) < 1e-3 and abs(errs[1] - 2.5e-3) < 1e-4
assert abs(errs[2] - 2.1e-6) < 1e-7 and abs(errs[3] - 1.6e-12) < 1e-13
# the data of Figure 9.2 (start 2.1)
x, e = 2.1, [2.1 - math.sqrt(2)]
for _ in range(4):
    x = 0.5 * (x + 2 / x); e.append(abs(x - math.sqrt(2)))
print("from 2.1:", [f"{v:.3g}" for v in e])
assert abs(e[1] - 0.112) < 1e-3 and abs(e[2] - 4.11e-3) < 1e-5 and abs(e[3] - 5.95e-6) < 1e-8

# Exercise 9.8
F = lambda p: torch.stack([p[0] ** 2 + p[1] ** 2 - 4, p[0] * p[1] - 1])
newton = lambda Fun, p: p - torch.linalg.solve(jacrev(Fun)(p), Fun(p))
a0 = torch.tensor([2.0, 0.5]); a1 = newton(F, a0)
print("a1 =", [round(v, 4) for v in a1.tolist()], "  F(a0) =", F(a0).tolist(),
      "  F(a1) =", [round(v, 4) for v in F(a1).tolist()])
assert torch.allclose(a1, torch.tensor([1.9333, 0.5167]), atol=5e-5)
assert torch.allclose(F(a1), torch.tensor([0.0047, -0.0011]), atol=5e-5)

errors: ['8.58e-02', '2.45e-03', '2.12e-06', '1.59e-12', '2.22e-16']
from 2.1: ['0.686', '0.112', '0.00411', '5.95e-06', '1.25e-11']
a1 = [1.9333, 0.5167]   F(a0) = [0.25, 0.0]   F(a1) = [0.0047, -0.0011]


In [10]:
# invariance: G = B o F o phi, phi(x) = o + A x
torch.manual_seed(4)
A = torch.randn(2, 2) + 2 * torch.eye(2); o = torch.tensor([0.3, -0.1]); B = torch.randn(2, 2) + 2 * torch.eye(2)
phi = lambda x: o + A @ x
G = lambda x: B @ F(phi(x))
x = torch.linalg.solve(A, a0 - o); p = a0.clone()
for k in range(5):
    x, p = newton(G, x), newton(F, p)
    assert torch.allclose(phi(x), p, atol=1e-10)
print("Newton iterates of B o F o phi are the images by phi^{-1} of those of F; limit:", p.tolist())

# local quadratic convergence with the constant beta * L of Theorem 9.12
astar = p
beta = op(torch.linalg.inv(jacrev(F)(astar)))
Lc = 2.0                    # DF(x) - DF(y) = [[2dx, 2dy], [dy, dx]], spectral norm exactly 2|x - y|
p = torch.tensor([2.0, 0.5]); e = [(p - astar).norm().item()]
for _ in range(4):
    p = newton(F, p); e.append((p - astar).norm().item())
print("beta =", round(beta, 4), "  errors:", [f"{v:.2e}" for v in e])
for k in range(3):
    if e[k + 1] > 1e-14:
        assert e[k + 1] <= beta * Lc * e[k] ** 2 * (1 + 1e-6)

Newton iterates of B o F o phi are the images by phi^{-1} of those of F; limit: [1.9318516525781366, 0.5176380902050416]
beta = 0.5988   errors: ['7.04e-02', '1.77e-03', '1.50e-06', '1.13e-12', '2.22e-16']


In [11]:
# Kantorovich's quantities for x^2 - 2 at a0 = 1 (Exercise 9.7) and for the system at a0
beta0, eta0, Lc = 0.5, 0.5, 2.0
print("x^2 - 2 at 1:  h0 =", beta0 * Lc * eta0)
x, xs = 1.0, [1.0]
for _ in range(8):
    x = 0.5 * (x + 2 / x); xs.append(x)
for k in range(8):
    assert abs(xs[k + 1] - xs[k]) <= eta0 / 2 ** k + 1e-15
    assert abs(xs[k] - math.sqrt(2)) <= eta0 / 2 ** (k - 1)

a0 = torch.tensor([2.0, 0.5])
J0 = jacrev(F)(a0)
beta0 = op(torch.linalg.inv(J0)); eta0 = torch.linalg.solve(J0, F(a0)).norm().item()
h0 = beta0 * 2.0 * eta0
print("system at (2, 1/2):  beta0 =", round(beta0, 4), " eta0 =", round(eta0, 4), " h0 =", round(h0, 4))
assert h0 <= 0.5
p, pts = a0.clone(), [a0.clone()]
for _ in range(6):
    p = newton(F, p); pts.append(p.clone())
for k in range(6):
    assert (pts[k + 1] - pts[k]).norm() <= eta0 / 2 ** k + 1e-15
    assert (pts[k] - astar).norm() <= eta0 / 2 ** (k - 1) + 1e-15
    assert (pts[k] - a0).norm() <= 2 * eta0
print("steps and errors obey the bounds eta0 / 2^k and eta0 / 2^(k-1); iterates stay in the ball")

# the pitfall: arctan
N = lambda x: x - (1 + x * x) * math.atan(x)
def iterate(x, n=3):
    out = []
    for _ in range(n):
        x = N(x); out.append(round(x, 4))
    return out
print("arctan from 1.0:", iterate(1.0), "   from 1.5:", iterate(1.5))
assert iterate(1.0) == [-0.5708, 0.1169, -0.0011] and iterate(1.5) == [-1.6941, 2.3211, -5.1141]
lo, hi = 1.0, 1.5
for _ in range(60):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if 2 * mid - (1 + mid * mid) * math.atan(mid) > 0 else (lo, mid)
print("x_c =", round(lo, 4), "   N(x_c) =", round(N(lo), 4))
assert abs(lo - 1.3917) < 1e-4 and abs(N(lo) + lo) < 1e-9

x^2 - 2 at 1:  h0 = 0.5
system at (2, 1/2):  beta0 = 0.5681  eta0 = 0.0687  h0 = 0.0781
steps and errors obey the bounds eta0 / 2^k and eta0 / 2^(k-1); iterates stay in the ball
arctan from 1.0: [-0.5708, 0.1169, -0.0011]    from 1.5: [-1.6941, 2.3211, -5.1141]
x_c = 1.3917    N(x_c) = -1.3917


## 4. Newton's method does not know that it is asked to minimize

Corollary 9.14: the iteration on `df = 0` converges quadratically to any
non-degenerate critical point, including a saddle.

In [12]:
fsad = lambda p: p[0] ** 3 - 3 * p[0] + p[1] ** 2            # minimum (1, 0), saddle (-1, 0)
step = lambda p: p - torch.linalg.solve(hessian(fsad)(p), jacrev(fsad)(p))
for start, target in ((torch.tensor([0.6, 0.4]), torch.tensor([1.0, 0.0])),
                      (torch.tensor([-0.6, 0.4]), torch.tensor([-1.0, 0.0]))):
    p, e = start.clone(), []
    for _ in range(6):
        p = step(p); e.append((p - target).norm().item())
    kind = "minimum" if target[0] > 0 else "saddle"
    print(f"start {start.tolist()} -> {kind} {target.tolist()}: errors", [f"{v:.1e}" for v in e])
    assert e[-1] < 1e-14

start [0.6, 0.4] -> minimum [1.0, 0.0]: errors ['1.3e-01', '7.8e-03', '3.1e-05', '4.7e-10', '0.0e+00', '0.0e+00']
start [-0.6, 0.4] -> saddle [-1.0, 0.0]: errors ['1.3e-01', '7.8e-03', '3.1e-05', '4.7e-10', '0.0e+00', '0.0e+00']


## Summary

* A bound on `|Df|` along a segment bounds `|f(b) - f(a)|`; there is no
  equality for vector-valued maps.
* The tangent affine map errs by at most `(L/2)|b - a|^2` when `Df` is
  `L`-Lipschitz.
* The rate of an iteration near a fixed point is `|Dg(a*)|`; gradient descent
  is a contraction with constant `max(|1 - eta mu|, |1 - eta L|)`, and a
  minimum sharper than `2/eta` repels it.
* Newton's method squares the error near a zero, is the same in all affine
  coordinates, converges to saddles as well as minima, and can diverge when
  started too far.